# ПР-9. Метод опорных векторов (SVM)

Исследовать масштабирование, ядра и параметрические компромиссы SVM для бинарной классификации.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Масштабирование и ядра

SVM чувствителен к масштабу признаков. Все преобразования помещайте в Pipeline, чтобы избежать утечки при CV.

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
pipe=Pipeline([('scale',StandardScaler()),('svc',SVC())])
search=GridSearchCV(pipe,{'svc__kernel':['linear','rbf'],'svc__C':[0.1,1,10],
                          'svc__gamma':['scale',0.01,0.1]},
                    scoring='f1_macro',cv=cv,n_jobs=-1)
search.fit(X_train,y_train)
pred=search.predict(X_test)
print('best:',search.best_params_,'CV macro F1:',round(search.best_score_,3))
print(classification_report(y_test,pred,target_names=['malignant','benign']))

## 2. Влияние C

Параметр C задаёт силу штрафа за ошибки: большее значение стремится точнее разделить train, но может хуже обобщать.

In [ ]:
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
cs=[0.01,0.1,1,10,100]
train_scores=[]; cv_scores=[]
for c in cs:
    m=Pipeline([('scale',StandardScaler()),('svc',SVC(C=c,kernel='rbf'))])
    m.fit(X_train,y_train)
    train_scores.append(f1_score(y_train,m.predict(X_train),average='macro'))
    cv_scores.append(cross_val_score(m,X_train,y_train,cv=cv,scoring='f1_macro').mean())
plt.semilogx(cs,train_scores,marker='o',label='train')
plt.semilogx(cs,cv_scores,marker='o',label='CV')
plt.xlabel('C'); plt.ylabel('F1 macro'); plt.legend(); plt.grid(True); plt.show()

## Контрольные вопросы и вывод

1. Зачем SVM требуется масштабирование признаков?
2. Как выбор ядра изменил результат и что означает RBF-ядро?
3. Как связаны C, сложность границы и риск переобучения?
4. Почему оценка на test не должна участвовать в подборе C?

**В отчёте:** лучшие параметры, метрики, график train/CV и объяснение выбора модели.